# Speech Transit Toolformer Colab

В этом ноутбуке отдельно демонстрируются четыре piepline с использованием кода из проекта, CLI-command и скриптов. Вся логика реализована в проекте.

## Клонируем проект

In [1]:
!git clone https://github.com/focus38/speech_toolformer.git

Cloning into 'speech_toolformer'...
remote: Enumerating objects: 606, done.
remote: Counting objects: 100% (606/606), done.
remote: Compressing objects: 100% (388/388), done.
remote: Total 606 (delta 258), reused 524 (delta 183), pack-reused 0 (from 0)
Receiving objects: 100% (606/606), 265.65 KiB | 1.92 MiB/s, done.
Resolving deltas: 100% (258/258), done.


In [3]:
!cd /content/speech_toolformer && git fetch && git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 377 bytes | 377.00 KiB/s, done.
From https://github.com/focus38/speech_toolformer
   3c0630f..ac76e7a  main       -> origin/main
Updating 3c0630f..ac76e7a
Fast-forward
 scripts/setup.sh | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


## Подготовка окружения

Для быстрого тестирования и только текста используйте `configs/fast_model.yaml`, где используется модель **Qwen2.5-3B-Instruct**.

Для эталонных запусков и мультимодальных задач используйте `configs/reference_model.yaml`. В этом конфигу используется модель **gemma-3n-E4B-it**.

Используйте GPU для запуска.

### Установка пакетов

In [4]:
!cd speech_toolformer/scripts && ./setup.sh

Setup complete.

Python: Python 3.12.13
Virtualenv: 0


In [10]:
%cd /content/speech_toolformer
!ls

/content/speech_toolformer
AGENTS.md  data  LICENSE    README.md  requirements.txt  specs	tests
configs    docs  notebooks  reports    scripts		 src


## Проверка и загрузка настроек

In [13]:
from src.utils.config import load_yaml_config

dataset_config = load_yaml_config('configs/dataset.yaml')
pipeline_config = load_yaml_config('configs/pipelines.yaml')
fast_pipeline_config = load_yaml_config('configs/fast_pipelines.yaml')
evaluation_config = load_yaml_config('configs/evaluation.yaml')
fast_model_config = load_yaml_config('configs/fast_model.yaml')
reference_model_config = load_yaml_config('configs/reference_model.yaml')

print('Text dataset output:', dataset_config['outputs']['text_dataset'])
print('Audio metadata output:', dataset_config['outputs']['audio_metadata'])
print('Fast text config:', fast_pipeline_config['common']['model_config_path'], fast_model_config['model']['id'])
print('Reference config:', pipeline_config['common']['model_config_path'], reference_model_config['model']['id'])

Text dataset output: data/synthetic_text/dataset.jsonl
Audio metadata output: data/synthetic_audio/metadata.jsonl
Fast text config: configs/fast_model.yaml Qwen/Qwen2.5-3B-Instruct
Reference config: configs/reference_model.yaml google/gemma-3n-E4B-it


## Загрузка текстового dataset

In [17]:
!mkdir /content/speech_toolformer/data/synthetic_text && gdown -O /content/speech_toolformer/data/synthetic_text --folder 1-wWwQgJSbsdJDhHaHVC4mgn05UwmK0JG

Retrieving folder contents
Processing file 1JaYSmFhd7nzx-S_l5GVlnyWXfCVdbOy4 dataset.jsonl
Processing file 1-SfCp9HInuqS_cBJ0-3uGVnJ2qz7edy2 test.jsonl
Processing file 17v2glpcMijh0Ter-fYc231P77Bpp_Z8w train.jsonl
Processing file 19PGngRGcPdBDjttE7Bd5o9erWegDOHgv validation.jsonl
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1JaYSmFhd7nzx-S_l5GVlnyWXfCVdbOy4
To: /content/speech_toolformer/data/synthetic_text/dataset.jsonl

Downloading...
From: https://drive.google.com/uc?id=1-SfCp9HInuqS_cBJ0-3uGVnJ2qz7edy2
To: /content/speech_toolformer/data/synthetic_text/test.jsonl

Downloading...
From: https://drive.google.com/uc?id=17v2glpcMijh0Ter-fYc231P77Bpp_Z8w
To: /content/speech_toolformer/data/synthetic_text/train.jsonl

Downloading...
From: https://drive.google.com/uc?id=19PGngRGcPdBDjttE7Bd5o9erWegDOHgv
To: /content/speech_toolformer/data/synthetic_text/validation.jsonl

Download

In [18]:
from src.data.loaders.jsonl import read_jsonl, write_jsonl

text_test_path = PROJECT_ROOT / dataset_config['outputs']['test']
if not text_test_path.exists():
  print("No dataset")
  #subprocess.run([str(PYTHON), '-m', 'src.cli', 'generate-text-dataset'], cwd=PROJECT_ROOT, check=True)
  #subprocess.run([str(PYTHON), '-m', 'src.cli', 'validate-dataset'], cwd=PROJECT_ROOT, check=True)
else:
  print('Using existing text dataset:', text_test_path)

test_rows = read_jsonl(text_test_path)
quick_rows = test_rows[:QUICK_N]
demo_dir = PROJECT_ROOT / 'data' / 'demo'
demo_dir.mkdir(parents=True, exist_ok=True)
quick_test_path = demo_dir / 'quick_test.jsonl'
write_jsonl(quick_test_path, quick_rows)
print(f'Fixed test split rows: {len(test_rows)}')
print(f'Quick demo rows: {len(quick_rows)} -> {quick_test_path}')

Using existing text dataset: /content/speech_toolformer/data/synthetic_text/test.jsonl
Fixed test split rows: 36
Quick demo rows: 5 -> /content/speech_toolformer/data/demo/quick_test.jsonl


## Загрузка аудио dataset

In [44]:
!rm -rf /content/speech_toolformer/data/synthetic_audio

In [45]:
!mkdir /content/speech_toolformer/data/synthetic_audio && gdown 1BidtCPHGXEVCkVKFoWGrKrF2KCGIoxAb -O /content/speech_toolformer/data/synthetic_audio/synthetic_audio.zip

Downloading...
From (original): https://drive.google.com/uc?id=1BidtCPHGXEVCkVKFoWGrKrF2KCGIoxAb
From (redirected): https://drive.google.com/uc?id=1BidtCPHGXEVCkVKFoWGrKrF2KCGIoxAb&confirm=t&uuid=fc806785-e270-43c6-a245-859b51778784
To: /content/speech_toolformer/data/synthetic_audio/synthetic_audio.zip



In [46]:
!unzip /content/speech_toolformer/data/synthetic_audio/synthetic_audio.zip -d /content/speech_toolformer/data/synthetic_audio/
!rm /content/speech_toolformer/data/synthetic_audio/synthetic_audio.zip

Archive:  /content/speech_toolformer/data/synthetic_audio/synthetic_audio.zip
   creating: /content/speech_toolformer/data/synthetic_audio/test/
  inflating: /content/speech_toolformer/data/synthetic_audio/test/ru_tool_0190.wav  
  inflating: /content/speech_toolformer/data/synthetic_audio/test/ru_tool_0198.wav  
  inflating: /content/speech_toolformer/data/synthetic_audio/test/ru_tool_0179.wav  
  inflating: /content/speech_toolformer/data/synthetic_audio/test/en_tool_0192.wav  
  inflating: /content/speech_toolformer/data/synthetic_audio/test/ru_tool_0188.wav  
  inflating: /content/speech_toolformer/data/synthetic_audio/test/en_tool_0181.wav  
  inflating: /content/speech_toolformer/data/synthetic_audio/test/ru_tool_0189.wav  
  inflating: /content/speech_toolformer/data/synthetic_audio/test/ru_tool_0193.wav  
  inflating: /content/speech_toolformer/data/synthetic_audio/test/en_tool_0202.wav  
  inflating: /content/speech_toolformer/data/synthetic_audio/test/ru_tool_0204.wav  
  inf

In [47]:
!cd /content/speech_toolformer/data/synthetic_audio && ls -la

total 92
drwxr-xr-x 5 root root  4096 Jul  2 03:37 .
drwxr-xr-x 5 root root  4096 Jul  2 03:37 ..
-rw-rw-r-- 1 root root 63021 Jun 30 08:00 metadata.jsonl
drwxrwxr-x 2 root root  4096 Jun 30 08:00 test
drwxrwxr-x 2 root root 12288 Jun 30 07:58 train
drwxrwxr-x 2 root root  4096 Jun 30 07:59 validation


In [48]:
audio_metadata_path = PROJECT_ROOT / dataset_config['outputs']['audio_metadata']
if RUN_FULL_EVALUATION and not audio_metadata_path.exists():
  print("No dataset")
  #subprocess.run([str(PYTHON), '-m', 'src.cli', 'generate-audio-dataset'], cwd=PROJECT_ROOT, check=True)
elif audio_metadata_path.exists():
  print('Using existing audio metadata:', audio_metadata_path)
else:
  print('Audio metadata is not present. Quick demo will synthesize tiny placeholder WAV files for the package pipeline runners.')

Using existing audio metadata: /content/speech_toolformer/data/synthetic_audio/metadata.jsonl


## Запуск в режиме quiсk демо для проверки окружения

In [12]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = Path.cwd()

if IN_COLAB and not (PROJECT_ROOT / 'configs').exists():
    # In a fresh Colab runtime, clone the repository or upload it before continuing.
    raise RuntimeError('Repository files are not present. Clone or upload the project, then rerun this cell.')

if not (PROJECT_ROOT / 'configs').exists():
    raise RuntimeError('Run from the repository root.')

PYTHON = PROJECT_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

INSTALL_DEPS = False
if INSTALL_DEPS:
    subprocess.run([str(PYTHON), '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=PROJECT_ROOT, check=True)

QUICK_DEMO = True
QUICK_N = 5
RUN_FULL_EVALUATION = False
print('Project root:', PROJECT_ROOT)
print('Python:', PYTHON)
print('Quick demo mode:', QUICK_DEMO)
print('Full fixed-split mode:', RUN_FULL_EVALUATION)

Project root: /content/speech_toolformer
Python: /usr/bin/python3
Quick demo mode: True
Full fixed-split mode: False


### Подготовка настроек


In [49]:
import soundfile as sf
import numpy as np
import yaml

quick_audio_metadata_path = demo_dir / 'quick_audio_metadata.jsonl'
quick_pipeline_config_path = demo_dir / 'quick_pipelines.yaml'

if QUICK_DEMO:
    quick_audio_dir = demo_dir / 'audio' / 'test'
    quick_audio_dir.mkdir(parents=True, exist_ok=True)
    metadata_rows = []
    for row in quick_rows:
        wav_path = quick_audio_dir / f"{row['id']}.wav"
        if not wav_path.exists():
            sf.write(wav_path, np.zeros(1600, dtype=np.float32), 16000)
        metadata_rows.append({
            'audio_path': wav_path.relative_to(PROJECT_ROOT).as_posix(),
            'sample_rate': 16000,
            'language': row['language'],
            'transcript': row['user_text'],
            'duration_seconds': 0.1,
            'tts_engine': 'quick_demo_placeholder',
            'speaker_id': f"quick_{row['language']}",
        })
    with quick_audio_metadata_path.open('w', encoding='utf-8') as file:
        for metadata_row in metadata_rows:
            file.write(json.dumps(metadata_row, ensure_ascii=False) + '\n')
    quick_pipeline_config = {
        'common': {
            'dataset_path': quick_test_path.relative_to(PROJECT_ROOT).as_posix(),
            'audio_metadata_path': quick_audio_metadata_path.relative_to(PROJECT_ROOT).as_posix(),
            'model_config_path': 'configs/fast_model.yaml',
            'predictions_dir': demo_dir.relative_to(PROJECT_ROOT).as_posix(),
        },
        'pipelines': {
            'A': {'name': 'text_to_tool', 'input_path': quick_test_path.relative_to(PROJECT_ROOT).as_posix(), 'output_path': 'data/demo/pipeline_a_predictions.jsonl'},
            'B': {'name': 'audio_to_transcript', 'input_path': quick_audio_metadata_path.relative_to(PROJECT_ROOT).as_posix(), 'output_path': 'data/demo/pipeline_b_predictions.jsonl'},
            'C': {'name': 'audio_to_transcript_and_tool', 'input_path': quick_audio_metadata_path.relative_to(PROJECT_ROOT).as_posix(), 'output_path': 'data/demo/pipeline_c_predictions.jsonl'},
            'D': {'name': 'audio_to_transcript_to_tool', 'input_path': quick_audio_metadata_path.relative_to(PROJECT_ROOT).as_posix(), 'transcript_output_path': 'data/demo/pipeline_d_transcripts.jsonl', 'output_path': 'data/demo/pipeline_d_predictions.jsonl'},
        },
    }
    quick_pipeline_config_path.write_text(yaml.safe_dump(quick_pipeline_config, sort_keys=False), encoding='utf-8')
    print('Quick audio metadata:', quick_audio_metadata_path)
    print('Quick pipeline config:', quick_pipeline_config_path)
else:
    print('Quick demo assets skipped.')

Quick audio metadata: /content/speech_toolformer/data/demo/quick_audio_metadata.jsonl
Quick pipeline config: /content/speech_toolformer/data/demo/quick_pipelines.yaml


### Pipeline A: text -> tool call

In [52]:
from src.models.inference.text_model import StubTextBackend, TextModelInference
from src.pipelines.pipeline_a.runner import run_pipeline_a

if QUICK_DEMO:
  print("Quick demo!")
  responses = []
  for row in quick_rows:
    if row.get('expected_tool_call'):
      responses.append(json.dumps({'tool_call': row.get('expected_tool_call')}, ensure_ascii=False))
    else:
      responses.append('No transport lookup needed.')

  text_inference = TextModelInference(
      backend=StubTextBackend(responses, model_name='quick-stub-text')
    )
  pipeline_a_records = run_pipeline_a(
      input_path=quick_test_path,
      output_path=demo_dir / 'pipeline_a_predictions.jsonl',
      inference=text_inference
    )
else:
  subprocess.run(
      [str(PYTHON),
      '-m',
      'src.cli',
      'run-pipeline-a',
      '--config',
      'configs/fast_pipelines.yaml'],
      cwd=PROJECT_ROOT,
      check=True
  )

print('\nPipeline A complete.')

Quick demo!


Processing pipeline A: 100%|██████████| 5/5 [00:00<00:00, 4835.49dataset item/s]


Pipeline A complete.


In [62]:
from src.data.loaders.jsonl import read_jsonl
import pandas as pd
from IPython.display import display, HTML

def read_and_display_jsonl(filepath, max_rows=None):
  records = read_jsonl(filepath)
  if not records:
    print("The file is empty")
    return None

  df = pd.DataFrame(records)
  if 'predicted_tool_call' in df.columns:
    df['tool_name'] = df['predicted_tool_call'].apply(
        lambda x: x.get('name', '') if isinstance(x, dict) else ''
    )
    df['tool_args'] = df['predicted_tool_call'].apply(
        lambda x: json.dumps(x.get('arguments', {}), ensure_ascii=False) if isinstance(x, dict) else ''
    )

  if 'latency_seconds' in df.columns:
    df['latency_ms'] = df['latency_seconds'].apply(lambda x: f"{x * 1000:.4f}" if pd.notna(x) else '')

  display_columns = [
      'example_id',
      'pipeline',
      'model_name',
      'prompt_version',
      'tool_name',
      'tool_args',
      'parse_status',
      'created_at'
  ]

  available_columns = [col for col in display_columns if col in df.columns]
  df_display = df[available_columns].copy()

  if max_rows:
    df_display = df_display.head(max_rows)

  pd.set_option('display.max_colwidth', 50)
  pd.set_option('display.width', None)

  return df_display

df = read_and_display_jsonl(demo_dir / 'pipeline_a_predictions.jsonl', max_rows=10)
df

,example_id,pipeline,model_name,prompt_version,tool_name,tool_args,parse_status,created_at
0,ru_tool_0190,A,quick-stub-text,tool_call_v1,transport.where_is_vehicle,"{""city"": ""moscow"", ""transport_type"": ""bus"", ""r...",ok,2026-07-02T03:42:01.581437Z
1,ru_tool_0198,A,quick-stub-text,tool_call_v1,transport.where_is_vehicle,"{""city"": ""paris"", ""transport_type"": ""trolleybu...",ok,2026-07-02T03:42:01.581734Z
2,ru_tool_0179,A,quick-stub-text,tool_call_v1,transport.where_is_vehicle,"{""city"": ""paris"", ""transport_type"": ""tram"", ""r...",ok,2026-07-02T03:42:01.581874Z
3,en_tool_0192,A,quick-stub-text,tool_call_v1,transport.where_is_vehicle,"{""city"": ""london"", ""transport_type"": ""trolleyb...",ok,2026-07-02T03:42:01.581981Z
4,ru_tool_0188,A,quick-stub-text,tool_call_v1,transport.where_is_vehicle,"{""city"": ""moscow"", ""transport_type"": ""bus"", ""r...",ok,2026-07-02T03:42:01.582083Z


In [84]:
# Single record
def print_single_pipelie_a_record(df: pd.DataFrame) -> None:
  record = df.loc[df['example_id'] == 'ru_tool_0190'].iloc[0]
  json_args = json.loads(record['tool_args'])
  html_output = f"""
  <div style="border: 1px solid #ddd; padding: 15px; border-radius: 5px; margin: 10px 0;">
      <h3 style="margin-top: 0; color: #2c3e50;">Example: {record['example_id']}</h3>
      <table style="width: 100%; border-collapse: collapse;">
          <tr><td style="padding: 5px; font-weight: bold; width: 200px;">Pipeline:</td><td style="padding: 5px;">{record['pipeline']}</td></tr>
          <tr><td style="padding: 5px; font-weight: bold;">Model:</td><td style="padding: 5px;">{record['model_name']}</td></tr>
          <tr><td style="padding: 5px; font-weight: bold;">Prompt Version:</td><td style="padding: 5px;">{record['prompt_version']}</td></tr>
          <tr><td style="padding: 5px; font-weight: bold;">Parse Status:</td><td style="padding: 5px;">{record['parse_status']}</td></tr>
          <tr><td style="padding: 5px; font-weight: bold;">Tool Name:</td><td style="padding: 5px;">{record['tool_name']}</td></tr>
          <tr><td style="padding: 5px; font-weight: bold;">Arguments:</td><td style="padding: 5px;">
          <pre style="margin: 0; background: #757575; padding: 10px; border-radius: 3px;">{json.dumps(json_args, indent=2, ensure_ascii=False)}</pre></td></tr>
          <tr><td style="padding: 5px; font-weight: bold;">Created At:</td><td style="padding: 5px;">{record['created_at']}</td></tr>
      </table>
  </div>
  """
  display(HTML(html_output))

print_single_pipelie_a_record(df)

Pipeline:,A
Model:,quick-stub-text
Prompt Version:,tool_call_v1
Parse Status:,ok
Tool Name:,transport.where_is_vehicle
Arguments:,"{ ""city"": ""moscow"", ""transport_type"": ""bus"", ""route_number"": ""40м"" }"
Created At:,2026-07-02T03:42:01.581437Z


### Pipeline B: audio -> transcript

In [ ]:
from src.models.inference.audio_model import AudioModelInference, StubAudioBackend, JointAudioOutput
from src.pipelines.pipeline_b.runner import run_pipeline_b

if QUICK_DEMO:
  transcripts = [row['user_text'] for row in quick_rows]
  audio_backend = StubAudioBackend(
      transcript_responses=transcripts,
      model_name='quick-stub-audio'
  )
  audio_inference_b = AudioModelInference(backend=audio_backend)
  pipeline_b_records = run_pipeline_b(
      dataset_path=quick_test_path,
      metadata_path=quick_audio_metadata_path,
      output_path=demo_dir / 'pipeline_b_predictions.jsonl',
      inference=audio_inference_b
  )
else:
  subprocess.run(
      [str(PYTHON),
      '-m',
       'src.cli',
       'run-pipeline-b',
       '--config',
       'configs/pipelines.yaml'],
      cwd=PROJECT_ROOT,
      check=True
  )

print('Pipeline B complete.')

### Pipeline C: audio -> transcript + tool call

In [ ]:
from src.pipelines.pipeline_c.runner import run_pipeline_c

if QUICK_DEMO:
  joint_responses = []
  for row in quick_rows:
      expected = row.get('expected_tool_call')
      raw_json = json.dumps({'tool_call': expected}, ensure_ascii=False) if expected else 'No transport lookup needed.'
      joint_response_item = JointAudioOutput(
          transcript=row['user_text'],
          raw_output=raw_json
      )
      joint_responses.append(joint_response_item)
  audio_inference_c = AudioModelInference(
      backend=StubAudioBackend(joint_responses=joint_responses, model_name='quick-stub-audio')
  )
  pipeline_c_records = run_pipeline_c(
      dataset_path=quick_test_path,
      metadata_path=quick_audio_metadata_path,
      output_path=demo_dir / 'pipeline_c_predictions.jsonl',
      inference=audio_inference_c
  )
else:
  subprocess.run(
      [str(PYTHON),
       '-m',
       'src.cli',
       'run-pipeline-c',
       '--config',
       'configs/pipelines.yaml'],
      cwd=PROJECT_ROOT,
      check=True
  )

print('Pipeline C complete.')

### Pipeline D: audio -> transcript -> tool call

In [ ]:
from src.pipelines.pipeline_d.runner import run_pipeline_d

if QUICK_DEMO:
  transcripts = [row['user_text'] for row in quick_rows]
  audio_inference_d = AudioModelInference(
        backend=StubAudioBackend(transcript_responses=transcripts, model_name='quick-stub-audio')
  )
  text_inference_d = TextModelInference(
      backend=StubTextBackend(responses, model_name='quick-stub-text')
  )
  pipeline_d_records = run_pipeline_d(
      dataset_path=quick_test_path,
      metadata_path=quick_audio_metadata_path,
      output_path=demo_dir / 'pipeline_d_predictions.jsonl',
      audio_inference=audio_inference_d,
      text_inference=text_inference_d
  )
else:
  subprocess.run(
      [str(PYTHON),
      '-m',
      'src.cli',
      'run-pipeline-d',
      '--config',
      'configs/pipelines.yaml'],
      cwd=PROJECT_ROOT,
      check=True
  )

print('Pipeline D complete.')

### Оценка результата

In [ ]:
from src.evaluation.benchmarks.evaluate_all import evaluate_all

if QUICK_DEMO:
  quick_eval_config_path = demo_dir / 'quick_evaluation.yaml'
  quick_eval_config = dict(evaluation_config)
  quick_eval_config['outputs'] = dict(evaluation_config['outputs'])
  quick_eval_config['outputs'].update({
      'metrics_dir': 'data/demo/metrics',
      'pipeline_a_metrics': 'data/demo/metrics/pipeline_a_metrics.json',
      'pipeline_b_metrics': 'data/demo/metrics/pipeline_b_metrics.json',
      'pipeline_c_metrics': 'data/demo/metrics/pipeline_c_metrics.json',
      'pipeline_d_metrics': 'data/demo/metrics/pipeline_d_metrics.json',
      'comparison_table': 'data/demo/metrics/comparison_table.csv',
      'failure_cases': 'data/demo/failure_cases.jsonl',
      'figures_dir': 'data/demo/figures',
  })
  quick_eval_config_path.write_text(
      yaml.safe_dump(quick_eval_config, sort_keys=False),
      encoding='utf-8'
  )
  evaluation_outputs = evaluate_all(
      pipeline_config_path=quick_pipeline_config_path,
      evaluation_config_path=quick_eval_config_path
  )
elif RUN_FULL_EVALUATION:
  subprocess.run(
      [str(PYTHON),
      '-m',
      'src.cli',
      'evaluate',
      '--config',
      'configs/evaluation.yaml'],
      cwd=PROJECT_ROOT,
      check=True
  )
  evaluation_outputs = {
      'comparison_table': PROJECT_ROOT / evaluation_config['outputs']['comparison_table'],
      'failure_cases': PROJECT_ROOT / evaluation_config['outputs']['failure_cases']
  }
else:
  evaluation_outputs = {}

print(evaluation_outputs)

## Метрики

In [ ]:
import pandas as pd

default_comparison_path = PROJECT_ROOT / evaluation_config['outputs']['comparison_table']
comparison_path = Path(evaluation_outputs.get('comparison_table', default_comparison_path))
if comparison_path.exists():
  metrics_table = pd.read_csv(comparison_path)
  display(metrics_table)
else:
  print('No comparison table yet.')

### Ошибки

In [ ]:
default_failure_path = PROJECT_ROOT / evaluation_config['outputs']['failure_cases']
failure_path = Path(evaluation_outputs.get('failure_cases', default_failure_path))

if failure_path.exists() and failure_path.stat().st_size > 0:
  failures = pd.read_json(failure_path, lines=True)
  display(failures.head(10))
else:
  print('No failure examples found, or evaluation has not been run.')

## Лучший pipeline

In [ ]:
if comparison_path.exists():
  metrics_table = pd.read_csv(comparison_path)
  tool_accuracy = metrics_table[metrics_table['metric'].eq('tool_exact_match_accuracy')].copy()
  wer = metrics_table[metrics_table['metric'].eq('wer')].copy()

  if not tool_accuracy.empty:
    best = tool_accuracy.sort_values('value', ascending=False).iloc[0]
    print(f"Best tool-calling pipeline: {best['pipeline']} with exact-match accuracy {best['value']:.3f}.")

  if not wer.empty:
    display(wer.sort_values('value'))

  print('For report conclusions, prefer the pipeline with the strongest tool accuracy after checking ASR WER and failure buckets.')
  print('Pipeline A is the clean-text baseline.')
  print('Pipelines C and D show audio-conditioned degradation and recovery.')
else:
  print('Run unified evaluation before selecting the best pipeline.')

## Запуск в режиме full evaluation

Установить `QUICK_DEMO = False` и `RUN_FULL_EVALUATION = True` в ячейке настройки, затем перезапустить ячейки в ноутбуке.

Для Pipeline A можно использовать `configs/fast_pipelines.yaml` для более быстрого тестирования текста.

Запуски референсной модели через конвиг `configs/pipelines.yaml`, который ссылается на configs/reference_model.yaml.

В полном режиме результаты, готовые для отчета, записываются в папку `data/metrics/`, `reports/failure_cases.jsonl`, and `reports/figures/`.
